# 01 · Ingesta → Bronze
Lee los CSV del lote desde el volumen, **comprueba el contrato de columnas**, normaliza nombres, descarta columnas
sensibles y añade metadatos (`lote_id`, `fecha_ingesta`, `archivo_origen`).

* Si falta una columna obligatoria, **la tarea falla aquí** y no se escribe nada (el job se detiene).
* Si el lote ya estaba cargado, se **sustituye** (re-ejecutar es seguro).

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from pyspark.sql import functions as F
from src.config import CATALOGO_POR_DEFECTO, VOLUMEN, T_BRONZE
from src.transformaciones import leer_csv_lote, preparar_bronze
from src.utilidades import escribir_lote

dbutils.widgets.text("catalogo", CATALOGO_POR_DEFECTO)
dbutils.widgets.text("lote", "lote_01")
catalogo, lote = dbutils.widgets.get("catalogo"), dbutils.widgets.get("lote")
tabla = f"{catalogo}.{T_BRONZE}"
ruta = f"/Volumes/{catalogo}/bronze/{VOLUMEN}/{lote}"
print(f"Lote: {lote}\nOrigen: {ruta}\nDestino: {tabla}")

## 1. Lectura (todo como texto, codificación latin-1) y contrato

In [0]:
origen = leer_csv_lote(spark, ruta)
filas_origen = origen.count()
print(f"Filas en el fichero: {filas_origen}")

# Lanza ErrorContrato (y falla la tarea) si falta una columna obligatoria
bronze, informe = preparar_bronze(origen, lote)
print("Columnas sensibles descartadas :", informe["sensibles"])
print("Columnas extra ignoradas (drift):", informe["extra"])

## 2. Escritura idempotente y comparación origen ↔ Bronze

In [0]:
escribir_lote(spark, bronze, tabla, lote)
filas_bronze = spark.table(tabla).filter(F.col("lote_id") == lote).count()
print(f"Origen: {filas_origen}  |  Bronze (lote {lote}): {filas_bronze}")
assert filas_origen == filas_bronze, "Se han perdido o duplicado filas en la ingesta"

display(spark.table(tabla).filter(F.col("lote_id") == lote).limit(5))